# Main Experiments: PPO, A2C, and REINFORCE

This notebook executes the primary experiments comparing PPO, A2C, and REINFORCE algorithms. It trains the models, saves them with associated metrics and videos, and supports result reloading across kernel restarts.

## File Overview

- `configs.py`: All experiment configurations
- `reward_v2.py`: Reward functions and metrics
- `envs.py`: Environment creation
- `train_sb3.py`: Training for PPO / A2C
- `reinforce.py`: REINFORCE baseline implementation
- `evaluate_and_record.py`: Evaluation, video recording, and file saving
- `generic_eval.py`: Common evaluation logic
- `video_utils.py`: Video display utilities

In [ ]:
from pathlib import Path
import sys, json, subprocess, os
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "configs.py").exists():
    BASE_DIR = PROJECT_DIR
else:
    BASE_DIR = Path("/mnt/data/highway_rl_suite")
    os.chdir(BASE_DIR)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
print("Working directory:", BASE_DIR)

In [ ]:
# !pip -q install stable-baselines3 gymnasium highway-env imageio moviepy

In [ ]:
from configs import EXPERIMENTS
EXPERIMENTS.keys()

## Global Parameters

Global parameters are set here. Start with smaller values to validate the pipeline, then scale up as needed.

In [ ]:
MAIN_TIMESTEPS = 120_000
MAIN_EVAL_EPISODES = 20
RUN_REINFORCE = True

SB3_RUNS = [
    ("ppo", "ppo_balanced"),
    ("ppo", "ppo_overtake"),
    ("ppo", "ppo_explore"),
    ("a2c", "a2c_baseline"),
]

## Training PPO / A2C

Train the PPO and A2C models.

In [ ]:
for algo, config_name in SB3_RUNS:
    cmd = [sys.executable, "train_sb3.py", "--algo", algo, "--config", config_name, "--timesteps", str(MAIN_TIMESTEPS)]
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)

## Training REINFORCE

Train the REINFORCE model.

In [ ]:
if RUN_REINFORCE:
    from configs import EXPERIMENTS
    from reinforce import ReinforceConfig, train_reinforce
    from envs import make_env
    cfg = EXPERIMENTS["reinforce_baseline"]["algo"]
    run_dir = BASE_DIR / "runs" / "reinforce_reinforce_baseline"
    run_dir.mkdir(parents=True, exist_ok=True)
    reinforce_cfg = ReinforceConfig(learning_rate=cfg["learning_rate"], gamma=cfg["gamma"], hidden_sizes=tuple(cfg["hidden_sizes"]), max_steps_per_episode=cfg["max_steps_per_episode"], batch_episodes=cfg["batch_episodes"])
    env_factory = lambda render_mode=None: make_env(EXPERIMENTS["reinforce_baseline"]["env"], EXPERIMENTS["reinforce_baseline"]["reward"], render_mode=render_mode)
    reinforce_agent = train_reinforce(env_factory=env_factory, episodes=cfg["episodes"], run_dir=run_dir, config=reinforce_cfg)

## Evaluate All Models and Record Videos

Evaluate all trained models and record a video for each.

In [ ]:
EVAL_JOBS = [
    ("ppo", "ppo_balanced", BASE_DIR / "runs" / "ppo_ppo_balanced" / "final_model.zip"),
    ("ppo", "ppo_overtake", BASE_DIR / "runs" / "ppo_ppo_overtake" / "final_model.zip"),
    ("ppo", "ppo_explore", BASE_DIR / "runs" / "ppo_ppo_explore" / "final_model.zip"),
    ("a2c", "a2c_baseline", BASE_DIR / "runs" / "a2c_a2c_baseline" / "final_model.zip"),
]
if RUN_REINFORCE:
    EVAL_JOBS.append(("reinforce", "reinforce_baseline", BASE_DIR / "runs" / "reinforce_reinforce_baseline" / "final_model.pt"))
for algo, config_name, model_path in EVAL_JOBS:
    if not model_path.exists():
        print("Model not found:", model_path)
        continue
    cmd = [sys.executable, "evaluate_and_record.py", "--algo", algo, "--config", config_name, "--model", str(model_path), "--episodes", str(MAIN_EVAL_EPISODES), "--deterministic"]
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)

## Load Saved Results

Load the saved results from the experiments.

In [ ]:
summary_rows = []
for algo, config_name, model_path in EVAL_JOBS:
    run_dir = model_path.parent
    eval_json = run_dir / "evaluation_summary.json"
    train_csv = run_dir / "train_metrics.csv"
    eval_csv = run_dir / "evaluation_episodes.csv"
    row = {"algo": algo, "config": config_name, "run_dir": str(run_dir)}
    if eval_json.exists():
        row.update(json.loads(eval_json.read_text()))
    row["train_csv_exists"] = train_csv.exists()
    row["eval_csv_exists"] = eval_csv.exists()
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_df

## Training Curves

Plot the training curves to analyze model performance.

In [ ]:
import matplotlib.pyplot as plt
for algo, config_name, model_path in EVAL_JOBS:
    train_csv = model_path.parent / "train_metrics.csv"
    if not train_csv.exists():
        continue
    df = pd.read_csv(train_csv)
    plt.figure(figsize=(8,4))
    plt.plot(df["episode"], df["episode_reward"])
    plt.title(f"Training reward — {algo} / {config_name}")
    plt.xlabel("Episode")
    plt.ylabel("Episode reward")
    plt.show()

## Compact Table for Report

Generate a compact summary table for the report.

In [ ]:
report_cols = ["algo", "config", "mean_return", "mean_episode_length", "mean_speed", "collision_rate", "mean_overtakes", "mean_overtaken_by_others", "mean_lane_changes", "mean_abs_acceleration"]
report_df = summary_df[[c for c in report_cols if c in summary_df.columns]].copy()
report_df.sort_values(["algo", "config"])

## Display an Evaluation Video

Display a sample evaluation video.

In [ ]:
from video_utils import show_video_file
video_path = BASE_DIR / "runs" / "ppo_ppo_overtake" / "videos" / "eval_episode_1.mp4"
if video_path.exists():
    display(show_video_file(video_path, width=800))
else:
    print("Video not found:", video_path)